# 🔍 Notebook 3 – YOLOv8 Object Detection Training

**Depends on:** `01_setup.ipynb` (must be run first in the same session)

**What this notebook does:**

| Step | Description | Output |
|------|-------------|--------|
| 3.1 | Load config | CONFIG dict |
| 3.2 | Build YOLO dataset from LineMOD | `Linemod_ready/images/` + `labels/` |
| 3.3 | Write `linemod_yolo.yaml` | YOLO config file |
| 3.4 | Train YOLOv8s | `runs/detect/linemod/weights/best.pt` |
| 3.5 | Evaluate + print metrics | Precision, Recall, mAP |
| 3.6 | Save best weights to Drive | `Drive/models/yolo_best.pt` |

**Next step → `04_pose_train.ipynb`**

## Cell 3.1 – Load shared config

In [ ]:
import os, json, sys, random, shutil
import numpy as np
import yaml
import pandas as pd
from PIL import Image as PILImage
from collections import defaultdict, Counter
from tqdm import tqdm

with open('/content/config.json') as f:
    CONFIG = json.load(f)

DATA_DIR    = CONFIG['DATA_DIR']
YOLO_DIR    = CONFIG['YOLO_DIR']
ALL_CLASSES = CONFIG['ALL_CLASSES']
CLASS_NAMES = CONFIG['CLASS_NAMES']
DRIVE_MODELS= CONFIG['DRIVE_MODELS']

random.seed(42)
print('✅ Config loaded.')

## Cell 3.2 – Build YOLO-format dataset from LineMOD

YOLO expects this directory layout:
```
Linemod_ready/
  images/train/*.png    ← copies of the RGB images
  images/val/*.png
  labels/train/*.txt    ← one .txt per image with YOLO bbox lines
  labels/val/*.txt
```

**YOLO label format** (per line): `class_id  cx  cy  w  h` — all values normalised 0–1.

Bounding boxes come from `gt.yml` field `obj_bb = [x, y, width, height]` (pixels). We convert to `cx/cy/w/h` normalised by image width and height.

Split ratio: **80% train / 20% val** (separate from the pose estimation split).
`class_map`: LineMOD ID (1-based, with gaps) → contiguous YOLO index (0-based).

In [ ]:
for sub in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    os.makedirs(os.path.join(YOLO_DIR, sub), exist_ok=True)

class_map = {cls_id: idx for idx, cls_id in enumerate(ALL_CLASSES)}

def save_yolo_sample(img_id, cls, yolo_idx, gt_entry, rgb_dir, split):
    """Copy one image and write its YOLO label file."""
    src_img = os.path.join(rgb_dir, f'{int(img_id):04d}.png')
    if not os.path.isfile(src_img):
        src_img = os.path.join(rgb_dir, f'{int(img_id):06d}.png')
    if not os.path.isfile(src_img):
        return False

    W, H   = PILImage.open(src_img).size
    x, y, bw, bh = gt_entry[0]['obj_bb']
    cx = (x + bw / 2) / W
    cy = (y + bh / 2) / H
    nw = bw / W
    nh = bh / H

    name    = f'{int(cls):02d}_{int(img_id):06d}'
    img_dst = os.path.join(YOLO_DIR, f'images/{split}', f'{name}.png')
    lbl_dst = os.path.join(YOLO_DIR, f'labels/{split}', f'{name}.txt')

    shutil.copy(src_img, img_dst)
    with open(lbl_dst, 'w') as f:
        f.write(f'{yolo_idx} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n')
    return True

stats = {}
for cls in ALL_CLASSES:
    gt_file  = os.path.join(DATA_DIR, cls, 'gt.yml')
    rgb_dir  = os.path.join(DATA_DIR, cls, 'rgb')
    yolo_idx = class_map[cls]

    if not os.path.isfile(gt_file):
        print(f'   ⚠️  Class {cls}: gt.yml missing')
        continue

    with open(gt_file) as f:
        gt = yaml.safe_load(f)

    img_ids = list(gt.keys())
    random.shuffle(img_ids)
    cut = int(0.8 * len(img_ids))

    n_train = sum(save_yolo_sample(i, cls, yolo_idx, gt[i], rgb_dir, 'train')
                  for i in img_ids[:cut])
    n_val   = sum(save_yolo_sample(i, cls, yolo_idx, gt[i], rgb_dir, 'val')
                  for i in img_ids[cut:])

    stats[cls] = {'train': n_train, 'val': n_val}
    print(f'   ✅ Class {cls:>2} ({CLASS_NAMES[int(yolo_idx)]}): '
          f'train={n_train}, val={n_val}')

print('\n📊 Summary:')
df = pd.DataFrame(stats).T.astype(int)
df.loc['TOTAL'] = df.sum()
print(df.to_string())
print('\n✅ YOLO dataset ready.')

## Cell 3.3 – Write the YOLO `data.yaml` configuration file

YOLOv8 requires a YAML config that tells it:
- `path` : absolute root directory of the dataset
- `train`: relative path to train images (from `path`)
- `val`  : relative path to val images
- `nc`   : number of classes
- `names`: list of class name strings

We write it dynamically so the `path` field always matches the current Colab session path — no manual editing needed.

In [ ]:
YOLO_YAML = '/content/linemod_yolo.yaml'

yaml_dict = {
    'path' : YOLO_DIR,
    'train': 'images/train',
    'val'  : 'images/val',
    'nc'   : len(ALL_CLASSES),
    'names': CLASS_NAMES,
}

with open(YOLO_YAML, 'w') as f:
    yaml.dump(yaml_dict, f, default_flow_style=False, sort_keys=False)

CONFIG['YOLO_YAML'] = YOLO_YAML
with open('/content/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

print(f'✅ YOLO config written to {YOLO_YAML}')
print(f'   Classes: {CLASS_NAMES}')

## Cell 3.4 – Train YOLOv8s on LineMOD

We start from `yolov8s.pt` (pretrained on COCO) and fine-tune for LineMOD.

**Key training arguments:**
| Argument | Value | Reason |
|----------|-------|--------|
| `epochs` | 50 | Converges well for LineMOD (~1000 imgs/class) |
| `imgsz` | 640 | Standard YOLO input resolution |
| `batch` | 16 (T4) / **64 (H100)** | Auto-scaled by GPU VRAM |
| `amp` | True | ~2× faster on modern GPUs |
| `patience` | 10 | Early stopping if val mAP doesn't improve |

All results (metrics, plots, weights) go to `runs/detect/linemod/`.

In [ ]:
from ultralytics import YOLO

import torch
yolo_model = YOLO('yolov8s.pt')

# H100 has 80 GB VRAM — use larger batch for faster training
_gpu       = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ''
_is_h100   = 'H100' in _gpu or 'H800' in _gpu
_batch     = 64 if _is_h100 else 16
print(f'   GPU: {_gpu}  →  batch={_batch}')

yolo_model.train(
    data         = YOLO_YAML,
    epochs       = 50,
    imgsz        = 640,
    batch        = _batch,
    device       = 0,
    amp          = True,
    patience     = 10,
    weight_decay = 0.0005,
    project      = 'runs/detect',
    name         = 'linemod',
)

print('\n✅ YOLOv8 training complete.')

## Cell 3.5 – Evaluate on validation set and show results

`model.val()` computes: **Precision (P)**, **Recall (R)**, **mAP@0.5**, **mAP@0.5:0.95**.

`plots=True` saves confusion matrix, PR curve, and prediction grid images to `runs/detect/linemod/`. We display one prediction grid below.

In [ ]:
from IPython.display import Image as IPImage, display

best_pt = 'runs/detect/linemod/weights/best.pt'
yolo_eval = YOLO(best_pt)

metrics = yolo_eval.val(
    data  = YOLO_YAML,
    plots = True,
    save  = True,
)

print('\n📊 YOLOv8 Validation Results')
print('─' * 40)
print(f'  Precision    : {metrics.box.mp:.4f}')
print(f'  Recall       : {metrics.box.mr:.4f}')
print(f'  mAP@0.5      : {metrics.box.map50:.4f}')
print(f'  mAP@0.5:0.95 : {metrics.box.map:.4f}')
print('─' * 40)

pred_grid = 'runs/detect/linemod/val_batch0_pred.jpg'
if os.path.isfile(pred_grid):
    print('\n🖼️  Sample predictions:')
    display(IPImage(pred_grid, width=900))

## Cell 3.6 – Save best YOLO weights to Google Drive

Colab sessions reset after ~12 hours (free tier) or ~24 hours (Pro). Saving to Drive ensures the weights survive the session.

We also update `config.json` with the path to the saved model so that `06_visualize.ipynb` can load it for detection.

In [ ]:
drive_yolo = os.path.join(DRIVE_MODELS, 'yolo_best.pt')
shutil.copy(best_pt, drive_yolo)

CONFIG['YOLO_MODEL_PATH'] = drive_yolo
with open('/content/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

print(f'✅ YOLO best weights saved to {drive_yolo}')

---
## ✅ YOLO Training Complete

- ✅ Dataset built at `Linemod_ready/`
- ✅ Model trained for up to 50 epochs with early stopping
- ✅ Best weights saved to Google Drive at `models/yolo_best.pt`
- ✅ `config.json` updated with model path

**Next step → Open and run `04_pose_train.ipynb`**